# 08 · Go-specific representation, objectives and initial learning

**Purpose.** Add a versioned spatial Go adapter around notebook 04's native Queen/Worker/Scout recurrence, with move energies, outcome, score and ownership predictions. This begins the **Go-trained AHR** route. The earlier CSP-only model, checkpoints and experiments remain a separate version.

**Inputs:** only tagged `ahr-export` definitions from notebooks 00–06 and this notebook's own verified generated development curriculum. No human games, external-engine teaching or final/lockbox shards. **Outputs:** reusable `GoAHR`, target/loss/search/loader definitions; `data/go08_*` curriculum; checkpointed warmup under `checkpoints/<run_id>/`; validation receipts under `reports/`. **Expected runtime:** default checks and short optimizer/recovery smokes take 1–3 minutes; full warmup is explicitly selected with `AHR08_MODE=train`. Checkpoint/resume instructions appear below.

The current CSP experiment showed no measured Go improvement. This notebook establishes learnable Go inputs and real outcome supervision, not a strong-player claim. Self-play follows in 09; promotion requires 11. The supplied theory's nested Q/W/S updates and bounded evaporation/deposition remain in `NativeAHR`. Spatial convolutions, supervised output heads and the search scheduling below are engineering extensions, not an asserted theorem that AHR solves Go.

In [ ]:
from pathlib import Path
import os, json, nbformat
PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks/00_instance_setup_and_recovery.ipynb").exists())
for number in range(7):
    path = next((PROJECT / "notebooks").glob(f"{number:02d}_*.ipynb"))
    book = nbformat.read(path,4)
    for index, cell in enumerate(book.cells):
        if cell.cell_type == "code" and "ahr-export" in cell.metadata.get("tags",[]):
            exec(compile(cell.source,f"{path.name}:export-{index}","exec"),globals())
print("Loaded definitions only; no upstream training/generation executed.")

## Spatial inputs and native AHR outputs

Twelve planes encode current/opponent stones, empty points, exact legal placement mask, current/opponent liberties, row/column coordinates, relative komi, pass count, player colour and move count. A small spatial encoder produces sixteen features per intersection plus a separate pass token. Each action is attached to its board location. Full superko history remains in the authoritative `GoState` and determines legal masks; the learned planes are a partial observation, not a replacement for the rule state.

The unchanged native recurrence processes these tokens and produces action energies. Separate learned heads predict signed outcome, raw score difference and ownership **from the player-to-move's perspective**. Relative move energy is never substituted for a game outcome value. Terminal states receive a dummy pass mask only to permit value diagnostics; search never emits a move from a terminal state.

In [ ]:
from dataclasses import dataclass, asdict
from collections import Counter
from contextlib import contextmanager
import math, time, copy, random, hashlib, tempfile, uuid, subprocess, sys, nbformat
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

GO_AHR_VERSION = "spatial-go-ahr-v1"

@dataclass(frozen=True)
class GoAHRConfig:
    board_size: int = 9
    spatial_channels: int = 32
    hidden_dim: int = 64
    queen_cycles: int = 2
    worker_steps: int = 2
    scout_steps: int = 2
    dropout: float = 0.1

    def __post_init__(self):
        if type(self.board_size) is not int or not 2 <= self.board_size <= 9:raise ValueError("Board size must be 2..9")
        if any(type(v) is not int or v<1 for v in (self.spatial_channels,self.hidden_dim,self.queen_cycles,self.worker_steps,self.scout_steps)):
            raise ValueError("Positive integer architecture sizes required")
        AHRConfig(hidden_dim=self.hidden_dim,iterations=self.queen_cycles,worker_steps=self.worker_steps,scout_steps=self.scout_steps,dropout=self.dropout,max_tokens=self.board_size**2+1)

def go08_source_hashes(project):
    result={}
    for number in (0,1,2,3,4,5,6,8):
        paths=list((Path(project)/"notebooks").glob(f"{number:02d}_*.ipynb"))
        if len(paths)!=1:raise ValueError(f"Expected one notebook {number:02d}")
        book=nbformat.read(paths[0],4)
        result[paths[0].name]=hashlib.sha256(json.dumps([c.source for c in book.cells if c.cell_type=="code" and "ahr-export" in c.metadata.get("tags",[])],ensure_ascii=False).encode()).hexdigest()
    return result

def go_model_digest(model):
    return architecture_parameter_digest(model)

def encode_go_batch(states,device="cpu"):
    if not states or any(not isinstance(s,GoState) for s in states):raise ValueError("Need GoState list")
    size=states[0].size
    if any(s.size!=size for s in states):raise ValueError("Mixed board sizes")
    planes=np.zeros((len(states),12,size,size),dtype=np.float32)
    masks=np.zeros((len(states),size*size+1),dtype=np.bool_)
    terminal=[]
    coords=np.linspace(-1,1,size,dtype=np.float32)
    for i,state in enumerate(states):
        board=np.array(state.board).reshape(size,size)
        planes[i,0]=board==state.to_play;planes[i,1]=board==-state.to_play;planes[i,2]=board==0
        legal=legal_actions(state)
        masks[i,list(legal) if legal else [state.pass_action]]=True
        planes[i,3]=masks[i,:size*size].reshape(size,size)
        for stones,liberties in groups_and_liberties(state.board,size):
            channel=4 if state.board[next(iter(stones))]==state.to_play else 5
            for point in stones:planes[i,channel].flat[point]=min(len(liberties),8)/8
        planes[i,6]=coords[:,None];planes[i,7]=coords[None,:]
        planes[i,8]=-state.to_play*state.komi/(size*size)
        planes[i,9]=state.consecutive_passes/2;planes[i,10]=state.to_play
        planes[i,11]=min(state.move_number,512)/512
        terminal.append(state.terminal)
    return {"planes":torch.as_tensor(planes,device=device),"legal_mask":torch.as_tensor(masks,device=device),
            "terminal":torch.tensor(terminal,dtype=torch.bool,device=device)}

class GoAHR(nn.Module):
    def __init__(self,config=None):
        super().__init__();self.config=config or GoAHRConfig();c=self.config
        self.spatial=nn.Sequential(nn.Conv2d(12,c.spatial_channels,3,padding=1),nn.SiLU(),
                                   nn.Conv2d(c.spatial_channels,c.spatial_channels,3,padding=1),nn.SiLU(),nn.Conv2d(c.spatial_channels,16,1))
        self.pass_embedding=nn.Parameter(torch.zeros(16))
        self.core=NativeAHR(AHRConfig(hidden_dim=c.hidden_dim,iterations=c.queen_cycles,worker_steps=c.worker_steps,
                                     scout_steps=c.scout_steps,dropout=c.dropout,max_tokens=c.board_size**2+1))
        role_dim=3*c.hidden_dim
        self.value_head=nn.Sequential(nn.Linear(role_dim,c.hidden_dim),nn.SiLU(),nn.Linear(c.hidden_dim,1))
        self.score_head=nn.Sequential(nn.Linear(role_dim,c.hidden_dim),nn.SiLU(),nn.Linear(c.hidden_dim,1))
        self.ownership_head=nn.Sequential(nn.Linear(16+role_dim,c.hidden_dim),nn.SiLU(),nn.Linear(c.hidden_dim,1))

    def forward(self,batch,queen_cycles=None):
        planes,legal=batch["planes"],batch["legal_mask"]
        n=self.config.board_size
        if planes.ndim!=4 or tuple(planes.shape[1:])!=(12,n,n) or legal.shape!=(planes.shape[0],n*n+1) or legal.dtype!=torch.bool:
            raise ValueError("Go batch shape/mask mismatch")
        if not bool(torch.isfinite(planes).all()) or not bool(legal.any(-1).all()):raise ValueError("Invalid Go features")
        spatial=self.spatial(planes).flatten(2).transpose(1,2)
        pass_token=spatial.mean(1,keepdim=True)+self.pass_embedding.reshape(1,1,16)
        tokens=torch.cat((spatial,pass_token),dim=1)
        core_batch={"x":tokens,"valid_mask":torch.ones_like(legal),"action_mask":legal}
        result=self.core(core_batch,iterations=queen_cycles)
        state=result["state"]
        roles=torch.cat([state[k] for k in ("queen","worker","scout")],-1)
        value=torch.tanh(self.value_head(roles).squeeze(-1))
        score=self.score_head(roles).squeeze(-1)*(n*n)
        owned=torch.cat((spatial,roles[:,None,:].expand(-1,n*n,-1)),dim=-1)
        ownership=torch.tanh(self.ownership_head(owned).squeeze(-1))
        if not all(bool(torch.isfinite(x).all()) for x in (value,score,ownership)):raise FloatingPointError("Nonfinite Go head")
        return {"policy_logits":result["policy_logits"],"energies":result["energies"],"value":value,"score":score,
                "ownership":ownership,"state":state,"trace":result["trace"],"strategic_status":"UNKNOWN"}

def terminal_go_targets(final_state,perspective):
    if not isinstance(final_state,GoState) or not final_state.terminal:raise ValueError("Outcome targets require actual two-pass terminal state")
    if type(perspective) is not int or perspective not in (-1,1):raise ValueError("Invalid perspective")
    owners=list(final_state.board);visited=set();adjacent=_neighbors(final_state.size)
    for point,stone in enumerate(final_state.board):
        if stone or point in visited:continue
        region={point};pending=[point];visited.add(point);borders=set()
        while pending:
            for near in adjacent[pending.pop()]:
                if final_state.board[near]:borders.add(final_state.board[near])
                elif near not in visited:visited.add(near);region.add(near);pending.append(near)
        owner=next(iter(borders)) if len(borders)==1 else 0
        for point in region:owners[point]=owner
    if sum(owners)-final_state.komi!=area_score(final_state):raise AssertionError("Ownership/score disagreement")
    relative=perspective*area_score(final_state)
    return {"outcome_target":float((relative>0)-(relative<0)),"score_target":float(relative),
            "ownership_target":[float(perspective*x) for x in owners],"terminal_targets_verified":True}

def go_learning_loss(model,examples):
    if not examples:raise ValueError("Empty training batch")
    device=next(model.parameters()).device;n=model.config.board_size
    states=[GoState.from_dict(e["state"]) for e in examples]
    batch=encode_go_batch(states,device);output=model(batch)
    policy_terms=[];value_terms=[];score_terms=[];ownership_terms=[]
    for i,e in enumerate(examples):
        weight=float(e.get("policy_weight",1.0))
        if not math.isfinite(weight) or weight<0:raise ValueError("Invalid policy weight")
        target=torch.tensor(e["policy_target"],device=device,dtype=torch.float32)
        if target.shape!=(n*n+1,) or not bool(torch.isfinite(target).all()) or bool((target<0).any()):raise ValueError("Invalid policy target")
        if weight:
            if states[i].terminal or abs(float(target.sum())-1)>1e-5 or bool((target[~batch["legal_mask"][i]]>0).any()):raise ValueError("Policy target must describe legal nonterminal actions")
            logp=F.log_softmax(output["policy_logits"][i],dim=-1)
            positive=target>0
            policy_terms.append(weight*(-target[positive]*logp[positive]).sum())
        supervision=[e.get(k) for k in ("outcome_target","score_target","ownership_target")]
        if any(v is not None for v in supervision) and e.get("terminal_targets_verified") is not True:
            raise ValueError("Unverified or cutoff result cannot label a Go outcome")
        outcome,score,ownership=supervision
        if outcome is not None:
            if outcome not in (-1,0,1):raise ValueError("Outcome must be -1,0,+1")
            value_terms.append((output["value"][i]-float(outcome)).square())
        if score is not None:
            if not math.isfinite(float(score)):raise ValueError("Invalid score target")
            score_terms.append(F.smooth_l1_loss(output["score"][i]/(n*n),torch.tensor(float(score)/(n*n),device=device)))
        if ownership is not None:
            t=torch.tensor(ownership,device=device,dtype=torch.float32)
            if t.shape!=(n*n,) or not bool(torch.isfinite(t).all()) or bool((t.abs()>1).any()):raise ValueError("Invalid ownership target")
            ownership_terms.append(F.mse_loss(output["ownership"][i],t))
    # Zero connections preserve the computation graph for heads without labels.
    zero=sum(x.sum()*0 for x in (output["value"],output["score"],output["ownership"],output["energies"]["soft"]))
    terms={"policy":torch.stack(policy_terms).mean() if policy_terms else zero,
           "value":torch.stack(value_terms).mean() if value_terms else zero,
           "score":torch.stack(score_terms).mean() if score_terms else zero,
           "ownership":torch.stack(ownership_terms).mean() if ownership_terms else zero}
    if not any((policy_terms,value_terms,score_terms,ownership_terms)):raise ValueError("No supervised targets")
    loss=terms["policy"]+terms["value"]+.5*terms["score"]+.5*terms["ownership"]
    if not bool(torch.isfinite(loss)):raise FloatingPointError("Nonfinite learning loss")
    return loss,{**{k:float(v.detach()) for k,v in terms.items()},"loss":float(loss.detach()),"examples":len(examples)}


## AHR-guided adaptive Go search

Queen governance increases search depth while time/node budgets permit; Workers apply authoritative legal transitions; Scouts order candidate moves with native AHR energies and look for opponent replies. Branch evidence is stored in bounded, decaying per-position/action pheromone during this search. Position keys include the entire rule state and superko history. Identical positions are evaluated once per search; sibling leaf evaluations are batched.

Negamax backups use the **separate learned outcome head**, with exact outcome only at actual two-pass terminals. The head starts untrained. Search targets are a declared soft distribution over backed-up explored root values; they are not proofs or claimed optimal policies. Beam restriction, depth, time and node cutoffs retain strategic UNKNOWN. Iterative deepening commits complete root iterations; a cutoff retains the last completed iteration or verified legal fallback. Deadlines are cooperative because an in-flight model call cannot be preempted. Final independent legality audit is included in elapsed time.

In [ ]:
@dataclass(frozen=True)
class GoAHRSearchConfig:
    max_depth: int = 3
    branch_width: int = 4
    max_nodes: int = 64
    queen_cycles: int = 1
    temperature: float = 1.0

    def __post_init__(self):
        if any(type(v) is not int or v<1 for v in (self.max_depth,self.branch_width,self.max_nodes,self.queen_cycles)):
            raise ValueError("Search limits must be positive integers")
        if self.max_depth>32 or not math.isfinite(self.temperature) or self.temperature<=0:raise ValueError("Invalid search depth/temperature")

class GoAHRDeadline(Exception):pass

def choose_go_ahr_move(state,model,time_budget_s=.25,search_config=None):
    config=search_config or GoAHRSearchConfig()
    if not isinstance(state,GoState) or not isinstance(model,GoAHR):raise TypeError("Expected GoState and GoAHR")
    if state.size!=model.config.board_size:raise ValueError("Model/board size mismatch")
    if model.training:raise ValueError("Set model.eval() before search")
    if isinstance(time_budget_s,bool) or not isinstance(time_budget_s,(float,int)) or not math.isfinite(time_budget_s) or time_budget_s<0:
        raise ValueError("Time budget must be finite and nonnegative")
    if config.queen_cycles>model.core.config.max_iterations:raise ValueError("Too many Queen cycles")
    start=time.monotonic();deadline=start+time_budget_s;device=next(model.parameters()).device
    cache={};pheromone={};stats={"nodes":0,"model_calls":0,"model_positions":0,"queen_iterations":0,"worker_transitions":0,"scout_candidates":0}
    def sync():
        if device.type=="cuda":torch.cuda.synchronize(device)
    def check():
        if time.monotonic()>=deadline:raise GoAHRDeadline("time_budget")
    def key(s):return hashlib.sha256(json.dumps(s.to_dict(),sort_keys=True,separators=(",",":")).encode()).hexdigest()
    def evaluate(states):
        pending=[];seen=set()
        for item in states:
            digest=key(item)
            if digest not in cache and digest not in seen:pending.append((digest,item));seen.add(digest)
        if not pending:return
        check()
        with torch.inference_mode():out=model(encode_go_batch([s for _,s in pending],device),queen_cycles=config.queen_cycles)
        sync();stats["model_calls"]+=1;stats["model_positions"]+=len(pending)
        values=out["value"].detach().cpu().tolist();energies=out["energies"]["soft"].detach().cpu().tolist()
        for i,(digest,item) in enumerate(pending):cache[digest]={"value":values[i],"energies":energies[i]}
        check()
    def visit(position,depth):
        check()
        if stats["nodes"]>=config.max_nodes:raise GoAHRDeadline("node_budget")
        stats["nodes"]+=1
        if position.terminal:return terminal_go_targets(position,position.to_play)["outcome_target"]
        evaluate([position]);digest=key(position)
        if depth==0:return cache[digest]["value"]
        legal=legal_actions(position)
        order=sorted(legal,key=lambda a:(cache[digest]["energies"][a]-pheromone.get((digest,a),0.0),a))
        width=min(len(order),config.branch_width+depth-1);selected=order[:width]
        stats["scout_candidates"]+=len(selected)
        children=[]
        for action in selected:
            check();child=play(position,action);stats["worker_transitions"]+=1;children.append((action,child))
        if depth==1:evaluate([child for _,child in children if not child.terminal])
        best=-float("inf")
        for action,child in children:
            value=-visit(child,depth-1);best=max(best,value)
            pheromone[(digest,action)]=float(np.clip(.8*pheromone.get((digest,action),0.0)+.1*value,-2,2))
        return best
    if state.terminal:
        return {"action":None,"policy_target":[0.0]*(state.pass_action+1),"root_value":terminal_go_targets(state,state.to_play)["outcome_target"],
                "status":"TERMINAL","strategic_status":"UNKNOWN","elapsed_s":time.monotonic()-start,**stats,"reason":"already_terminal","fallback_pass":False}
    fallback=state.pass_action;selected_action=fallback;root_values=None;reason="zero_budget";depth_done=0;fallback_used=True
    if time_budget_s>0:
        try:
            evaluate([state]);root_key=key(state);legal=legal_actions(state)
            order=sorted(legal,key=lambda a:(cache[root_key]["energies"][a],a))
            # A valid learned policy is usable if a deeper iteration later runs out.
            selected_action=order[0];reason="policy_only";fallback_used=False
            for depth in range(1,config.max_depth+1):
                check();iteration={};stats["queen_iterations"]+=1
                order=sorted(legal,key=lambda a:(cache[root_key]["energies"][a]-pheromone.get((root_key,a),0.0),a))
                chosen=order[:min(len(order),config.branch_width+depth-1)]
                stats["scout_candidates"]+=len(chosen)
                children=[]
                for action in chosen:
                    check();children.append((action,play(state,action)));stats["worker_transitions"]+=1
                if depth==1:evaluate([child for _,child in children if not child.terminal])
                for action,child in children:
                    iteration[action]=-visit(child,depth-1)
                check()
                root_values=iteration;depth_done=depth
                selected_action=max(iteration,key=lambda a:(iteration[a],-a));reason="bounded_search"
                for action,value in iteration.items():pheromone[(root_key,action)]=float(np.clip(.8*pheromone.get((root_key,action),0.0)+.1*value,-2,2))
        except GoAHRDeadline as cutoff:
            reason=str(cutoff)
    policy=np.zeros(state.pass_action+1,dtype=float)
    if fallback_used:
        policy[selected_action]=1.;root_value=None
    elif root_values:
        moves=sorted(root_values);values=np.array([root_values[a] for a in moves],dtype=float)/config.temperature
        probabilities=np.exp(values-values.max());probabilities/=probabilities.sum();policy[moves]=probabilities
        root_value=float(max(root_values.values()))
    elif cache:
        root=cache.get(key(state))
        if root:
            legal=legal_actions(state);scores=-np.array([root["energies"][a] for a in legal])/config.temperature
            probs=np.exp(scores-scores.max());probs/=probs.sum();policy[list(legal)]=probs;root_value=float(root["value"])
        else:policy[selected_action]=1.;root_value=None
    else:policy[selected_action]=1.;root_value=None
    child=play(state,selected_action)
    if not verify_go_emission(state,selected_action,child):raise AssertionError("Independent final move verification failed")
    sync();elapsed=time.monotonic()-start;missed=elapsed>time_budget_s
    return {"action":selected_action,"policy_target":policy.tolist(),"root_value":root_value,"status":"UNKNOWN" if reason in ("time_budget","node_budget","zero_budget") or missed else "LEGAL_MOVE",
            "strategic_status":"UNKNOWN","elapsed_s":elapsed,**stats,"reason":"final_audit_overrun" if missed and reason=="bounded_search" else reason,
            "completed_depth":depth_done,"deadline_missed":missed,"fallback_pass":fallback_used,"pheromone_entries":len(pheromone),"verified_legal":True,
            "policy_target_kind":"softmax_explored_backed_up_values" if root_values else "native_energy_policy" if cache else "verified_fallback"}

def _go_run_dir(project,run_id):
    if not isinstance(run_id,str) or not run_id or Path(run_id).name!=run_id or run_id in (".",".."):raise ValueError("Simple run ID required")
    return Path(project)/"checkpoints"/run_id

def strict_go_checkpoint(project,run_id,selection="latest"):
    root=_go_run_dir(project,run_id);path=root/"manifest.json"
    if selection not in ("latest","best"):raise ValueError("Select latest or best")
    if not path.exists():
        if list(root.glob("*.pt")):raise ValueError("Payload exists without manifest; refusing restart")
        return None
    manifest=json.loads(path.read_text());entries=manifest.get("entries",[])
    if manifest.get("schema_version")!=1 or not entries:raise ValueError("Invalid manifest")
    if len({e["file"] for e in entries})!=len(entries) or len({e["global_step"] for e in entries})!=len(entries):raise ValueError("Duplicate manifest entries")
    if manifest.get("latest")!=max(entries,key=lambda e:e["global_step"])["file"]:raise ValueError("Manifest would rewind")
    states={}
    for entry in entries:
        state=_checkpoint_entry_state(root,entry)
        if state is None:raise ValueError("Corrupt committed checkpoint")
        states[entry["file"]]=state
    if manifest.get(selection) not in states:raise ValueError("Missing checkpoint selection")
    return states[manifest[selection]]

def load_go_ahr(project,run_id,device="cpu",selection="best",allow_validation=False):
    state=strict_go_checkpoint(project,run_id,selection)
    if state is None:raise FileNotFoundError(run_id)
    if state.get("schema") not in ("ahr_go_learning_v1","ahr_go_selfplay_v1"):raise ValueError("Wrong Go model checkpoint schema")
    provenance=state.get("provenance",{})
    if provenance.get("trained_on_go") is not True or provenance.get("training_route") not in ("go_supervised","go_selfplay"):
        raise ValueError("Missing Go learning provenance")
    if provenance.get("purpose")!="go_training" or state.get("optimizer_step",0)<=0:
        if not allow_validation:raise ValueError("Validation/untrained snapshots cannot represent a trained Go player")
    if state.get("source_hashes")!=go08_source_hashes(project):raise ValueError("Go architecture sources changed; explicit migration required")
    model=GoAHR(GoAHRConfig(**state["model_config"]));model.load_state_dict(state["model"],strict=True);model.to(device).eval()
    root=_go_run_dir(project,run_id);manifest=json.loads((root/"manifest.json").read_text());entry=next(e for e in manifest["entries"] if e["file"]==manifest[selection])
    metadata={**provenance,"provenance":provenance,"run_id":run_id,"selection":selection,"schema":state["schema"],
              "checkpoint_file":str((root/entry["file"]).relative_to(Path(project))),"checkpoint_sha256":entry["sha256"],
              "model_config":state["model_config"],"model_sha256":go_model_digest(model),"optimizer_step":state["optimizer_step"],
              "source_hashes":state["source_hashes"],"candidate_only":True}
    return model,metadata


## Verified warmup curriculum

The bounded generator records legal placement games, then explicitly plays two passes and checks terminal ownership against the independent rules score. Earlier positions receive outcomes of that *recorded continuation*, never a claim about best play. Optional policy labels identify legal moves that maximize **immediate captures** among enumerated legal moves; they are tactical objectives, not optimal Go moves. Rows without a capture target have zero policy loss weight. No labels are derived from a move cap: the recorded passes are actual moves in these deliberately constructed warmup games.

Training/development groups are assigned at the whole-game level before label generation. Equivalent board/history states under the eight board symmetries cannot cross groups/splits; duplicates are removed before writing. Source replay moves, terminal state and hashes make every outcome target auditable. This small warmup is not a comprehensive Go curriculum. Self-play in 09 supplies the continuing game distribution; final evaluation remains separate.

Sampling uses adjacent ply pairs so both players appear in every split. The receipt reports turn and signed-outcome counts; this avoids an earlier all-even-ply sampling bias. Recorded random continuations can still have weak or biased outcome distributions, so low warmup loss is not evidence of strategic value calibration.

In [ ]:
def _go_canonical_state(state):
    # Reuse03's exact symmetry transform; omit move_number only for duplicate detection.
    forms=[]
    for symmetry in range(8):
        value=transform_go(state,symmetry).to_dict();value.pop("move_number")
        forms.append(json.dumps(value,sort_keys=True,separators=(",",":")))
    return hashlib.sha256(min(forms).encode()).hexdigest()

def make_go_warmup_curriculum(project,board_size=9,seed=20261001,games=10,placements=48):
    if type(games) is not int or games<4 or games>256 or type(placements) is not int or not 4<=placements<=200:
        raise ValueError("Bounded generator: 4..256 games, 4..200 placements")
    data={"train":[],"dev":[]};games_log=[];seen=set();candidate_counts=Counter()
    dev_start=max(1,int(.8*games))
    for game_index in range(games):
        split="train" if game_index<dev_start else "dev"
        group=f"go08-game-{seed}-{game_index}";rng=random.Random(seed+game_index)
        state=initial_state(board_size,7.5 if board_size==9 else .5);history=[state];moves=[]
        for ply in range(placements):
            options=[a for a in legal_actions(state) if a!=state.pass_action]
            if not options:break
            action=rng.choice(options);child=play(state,action)
            if not verify_go_emission(state,action,child):raise AssertionError("Generated illegal transition")
            moves.append(action);state=child;history.append(state)
        for _ in range(2):
            child=play(state,state.pass_action)
            if not verify_go_emission(state,state.pass_action,child):raise AssertionError("Pass verification failed")
            moves.append(state.pass_action);state=child
        final=state
        game_log={"group_id":group,"split":split,"initial":history[0].to_dict(),"actions":moves,"terminal":final.to_dict()}
        games_log.append(game_log)
        anchors=[max(1,len(history)//4),max(1,len(history)//2),max(1,3*len(history)//4),len(history)-1]
        # Adjacent plies deliberately include both players; regular even sampling does not.
        chosen=sorted({j for i in anchors for j in (i,i+1 if i+1<len(history) else i-1) if 0<=j<len(history)})
        for index in chosen:
            position=history[min(index,len(history)-1)];canonical=_go_canonical_state(position)
            if canonical in seen:continue
            seen.add(canonical);legal=legal_actions(position);captures={}
            enemy_before=sum(x==-position.to_play for x in position.board)
            for action in legal:
                child=play(position,action)
                captures[action]=enemy_before-sum(x==-position.to_play for x in child.board)
            maximum=max(captures.values());policy=[0.0]*(board_size*board_size+1)
            winners=[a for a,c in captures.items() if c==maximum] if maximum>0 else []
            for action in winners:policy[action]=1/len(winners)
            targets=terminal_go_targets(final,position.to_play)
            example={"state":position.to_dict(),"policy_target":policy,"policy_weight":float(bool(winners)),**targets,
                     "group_id":group,"split":split,"canonical_state":canonical,"source":"verified_recorded_continuation",
                     "policy_target_kind":"maximum_immediate_captures" if winners else "no_policy_label",
                     "max_immediate_captures":maximum,"continuation_index":index,"terminal_digest":hashlib.sha256(json.dumps(final.to_dict(),sort_keys=True).encode()).hexdigest()}
            data[split].append(example);candidate_counts[split+"_rows"]+=1;candidate_counts[split+"_capture_rows"]+=bool(winners)
            candidate_counts[split+("_black_to_move" if position.to_play==BLACK else "_white_to_move")]+=1
            candidate_counts[split+"_outcome_"+str(int(targets["outcome_target"]))]+=1
    if not data["train"] or not data["dev"]:raise ValueError("Generator produced empty split")
    if any({GoState.from_dict(r["state"]).to_play for r in data[split]}!={BLACK,WHITE} for split in ("train","dev")):
        raise ValueError("Warmup must cover both players to move in each split")
    payload={"schema":"ahr_go_warmup_data_v1",**data,"games":games_log,
             "config":{"board_size":board_size,"seed":seed,"games":games,"placements":placements},
             "provenance":{"training_route":"go_supervised","trained_on_go":True,"purpose":"go_training","sources":go08_source_hashes(project)},
             "counts":dict(candidate_counts)}
    digest=hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()
    path=Path(project)/"data"/f"go08_warmup_{digest[:12]}.json";atomic_json(path,payload)
    return path,payload

def verify_go_warmup_data(data):
    if data.get("schema")!="ahr_go_warmup_data_v1":raise ValueError("Wrong warmup schema")
    groups={};seen=set()
    for record in data["games"]:
        if record["group_id"] in groups:raise ValueError("Duplicate game group")
        state=GoState.from_dict(record["initial"]);history=[state]
        for action in record["actions"]:
            successor=play(state,action)
            if not verify_go_emission(state,action,successor):raise ValueError("Replay verification failed")
            state=successor;history.append(state)
        if not state.terminal or state.to_dict()!=record["terminal"]:raise ValueError("Incomplete/unmatched terminal replay")
        groups[record["group_id"]]=(record,history,state)
    for split in ("train","dev"):
        if not data[split]:raise ValueError("Empty split")
        for row in data[split]:
            record,history,terminal=groups[row["group_id"]]
            if row["split"]!=split or record["split"]!=split:raise ValueError("Group/split mismatch")
            position=GoState.from_dict(row["state"])
            if history[row["continuation_index"]].to_dict()!=row["state"]:raise ValueError("Position not in claimed replay")
            expected=terminal_go_targets(terminal,position.to_play)
            if any(row.get(k)!=v for k,v in expected.items()):raise ValueError("Invalid terminal supervision")
            canonical=_go_canonical_state(position)
            if canonical!=row["canonical_state"] or canonical in seen:raise ValueError("Equivalent state duplicated across curriculum")
            seen.add(canonical)
            before=sum(x==-position.to_play for x in position.board)
            captures={a:before-sum(x==-position.to_play for x in play(position,a).board) for a in legal_actions(position)}
            maximum=max(captures.values());winners=[a for a,v in captures.items() if v==maximum] if maximum>0 else []
            target=[0.0]*(position.pass_action+1)
            for action in winners:target[action]=1/len(winners)
            if row["policy_target"]!=target or row["policy_weight"]!=float(bool(winners)):raise ValueError("Invalid bounded tactical label")
    return True

@dataclass(frozen=True)
class GoWarmupConfig:
    epochs:int=12
    batch_size:int=8
    learning_rate:float=.001
    seed:int=20261002
    device:str="cuda:0"
    checkpoint_steps:int=50
    checkpoint_seconds:float=600.
    purpose:str="go_training"
    def __post_init__(self):
        if any(type(v) is not int or v<1 for v in (self.epochs,self.batch_size,self.checkpoint_steps)) or self.checkpoint_steps>250:
            raise ValueError("Invalid warmup integer settings")
        if not 0<self.learning_rate<=.1 or not 0<self.checkpoint_seconds<=900 or self.purpose not in ("go_training","synthetic_smoke"):
            raise ValueError("Invalid warmup settings")
        if self.device not in ("cpu","cuda:0","cuda:1"):raise ValueError("Supported devices:cpu,cuda:0,cuda:1")

def evaluate_go_examples(model,examples,batch_size=8):
    training=model.training;model.eval();sums=Counter()
    with torch.inference_mode():
        for start in range(0,len(examples),batch_size):
            rows=examples[start:start+batch_size];_,metrics=go_learning_loss(model,rows)
            for key in ("loss","policy","value","score","ownership"):sums[key]+=metrics[key]*len(rows)
    model.train(training)
    return {k:v/len(examples) for k,v in sums.items()}

def _go_warmup_order(size,seed,epoch):
    return torch.randperm(size,generator=torch.Generator().manual_seed(seed+epoch)).tolist()

def run_go_warmup(project,run_id,data_file,config=None,model_config=None,stop_after_steps=None):
    project=Path(project);config=config or GoWarmupConfig();model_config=model_config or GoAHRConfig()
    data_path=Path(data_file)
    if not data_path.is_absolute():data_path=project/data_path
    data=json.loads(data_path.read_text());verify_go_warmup_data(data)
    if data["config"]["board_size"]!=model_config.board_size:raise ValueError("Curriculum/model size mismatch")
    device=torch.device(config.device)
    if device.type=="cuda" and (not torch.cuda.is_available() or torch.cuda.device_count()<=device.index):raise ValueError("Requested GPU unavailable")
    total_steps=config.epochs*math.ceil(len(data["train"])/config.batch_size)
    if config.purpose=="synthetic_smoke" and total_steps>16:raise ValueError("Smoke must remain short")
    identity={"train":asdict(config),"model":asdict(model_config),"data_file":str(data_path.resolve().relative_to(project.resolve())),
              "data_sha256":file_sha256(data_path),"sources":go08_source_hashes(project),"torch":str(torch.__version__),"cuda":torch.version.cuda,
              "gpu":torch.cuda.get_device_name(device) if device.type=="cuda" else None}
    if config.purpose=="go_training":
        receipt_path=project/"reports/08_exact_resume_validation.json"
        receipts=json.loads(receipt_path.read_text()) if receipt_path.exists() else {}
        checks=receipts.get("configurations",[])
        if not any(r.get("status")=="PASS" and r.get("source_hashes")==identity["sources"] and r.get("device")==config.device and r.get("model_config")==asdict(model_config) for r in checks):
            raise ValueError("Matching fresh-process resume smoke required before warmup")
    torch.set_num_threads(1);torch.use_deterministic_algorithms(True);torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.allow_tf32=False;torch.backends.cuda.matmul.allow_tf32=False
    random.seed(config.seed);np.random.seed(config.seed);torch.manual_seed(config.seed)
    model=GoAHR(model_config).to(device);optimizer=torch.optim.AdamW(model.parameters(),lr=config.learning_rate,weight_decay=1e-4)
    scheduler=torch.optim.lr_scheduler.ExponentialLR(optimizer,gamma=.97);scaler=torch.amp.GradScaler("cuda",enabled=False)
    root=_go_run_dir(project,run_id)
    with checkpoint_lock(root):
        state=strict_go_checkpoint(project,run_id)
        epoch=cursor=step=0;sequence=-1;best=float("inf");metrics=[];order=_go_warmup_order(len(data["train"]),config.seed,0)
        if state:
            if state.get("schema")!="ahr_go_learning_v1" or state["config"]!=identity:raise ValueError("Warmup resume identity mismatch")
            model.load_state_dict(state["model"]);optimizer.load_state_dict(state["optimizer"]);scheduler.load_state_dict(state["scheduler"]);scaler.load_state_dict(state["scaler"])
            epoch=state["sampler"]["epoch"];cursor=state["sampler"]["cursor"];order=state["sampler"]["order"]
            step=state["optimizer_step"];sequence=state["global_step"];best=state["best_dev_loss"];metrics=state["metrics"]
            if order!=_go_warmup_order(len(data["train"]),config.seed,epoch) or not 0<=cursor<len(order):raise ValueError("Bad sampler state")
            restore_rng(state["rng"])
        resumed_from=step;last_step=step;last_time=time.monotonic()
        def save(stage,score=None):
            nonlocal sequence,best,last_step,last_time
            improve=score is not None and score<best
            if improve:best=score
            sequence+=1
            snapshot={"schema":"ahr_go_learning_v1","model":architecture_cpu_tree(model.state_dict()),"model_config":asdict(model_config),
                      "optimizer":architecture_cpu_tree(optimizer.state_dict()),"scheduler":scheduler.state_dict(),"scaler":scaler.state_dict(),
                      "rng":capture_rng(),"sampler":{"epoch":epoch,"cursor":cursor,"order":order},"global_step":sequence,"optimizer_step":step,
                      "config":identity,"source_hashes":identity["sources"],"metrics":metrics,"stage":stage,"best_dev_loss":best,
                      "provenance":{"training_route":"go_supervised","trained_on_go":True,"purpose":config.purpose,"candidate_only":True,
                                    "dataset_sha256":identity["data_sha256"],"target_scope":"recorded terminal continuation and bounded immediate captures"},
                      "replay":None,"distributed":{"world_size":1,"device":config.device}}
            _save_checkpoint_unlocked(root,snapshot,best=improve,resume_cell="08: Launch/resume with the same AHR08_RUN_ID and configuration")
            atomic_json(root/"progress.json",{"stage":stage,"optimizer_step":step,"epoch":epoch,"cursor":cursor,"resumed_from":resumed_from})
            last_step=step;last_time=time.monotonic()
        if not state:
            metrics.append({"kind":"initial","optimizer_step":0,"development":evaluate_go_examples(model,data["dev"],config.batch_size)})
            save("initialized")
        while epoch<config.epochs and (stop_after_steps is None or step<stop_after_steps):
            rows=[data["train"][i] for i in order[cursor:cursor+config.batch_size]]
            model.train();optimizer.zero_grad(set_to_none=True);loss,train_metrics=go_learning_loss(model,rows)
            loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.,error_if_nonfinite=True);optimizer.step()
            step+=1;cursor+=len(rows)
            if cursor>=len(order):
                epoch+=1;cursor=0;order=_go_warmup_order(len(data["train"]),config.seed,epoch);scheduler.step()
                dev=evaluate_go_examples(model,data["dev"],config.batch_size)
                metrics.append({"kind":"epoch","epoch":epoch,"optimizer_step":step,"training_last_batch":train_metrics,"development":dev})
                save("epoch",dev["loss"])
            elif step-last_step>=config.checkpoint_steps or time.monotonic()-last_time>=config.checkpoint_seconds:save("periodic")
        if step!=last_step or epoch>=config.epochs and (state is None or state.get("stage")!="complete"):
            save("complete" if epoch>=config.epochs else "interrupted_at_requested_step")
        return {"run_id":run_id,"complete":epoch>=config.epochs,"optimizer_step":step,"resumed_from":resumed_from,"metrics":metrics}

GO08_BOOTSTRAP = r"""
import sys,json,pathlib,nbformat
project=pathlib.Path(sys.argv[1])
for number in (*range(7),8):
    path=next((project/'notebooks').glob(f'{number:02d}_*.ipynb'))
    for i,c in enumerate(nbformat.read(path,4).cells):
        if c.cell_type=='code' and 'ahr-export' in c.metadata.get('tags',[]):exec(compile(c.source,str(path)+':export-'+str(i),'exec'),globals())
args=json.loads(pathlib.Path(sys.argv[2]).read_text())
result=run_go_warmup(project,args['run_id'],args['data_file'],GoWarmupConfig(**args['config']),GoAHRConfig(**args['model_config']),args.get('stop_after_steps'))
atomic_json(project/'reports'/args['result_file'],result)
"""

def _go08_fresh_launch(project,run_id,data_file,config,model_config,stop=None):
    project=Path(project);launch=project/"reports"/f"08-launch-{uuid.uuid4().hex}.json";result_file=f"08-result-{uuid.uuid4().hex}.json"
    args={"run_id":run_id,"data_file":str(data_file),"config":asdict(config),"model_config":asdict(model_config),"stop_after_steps":stop,"result_file":result_file}
    atomic_json(launch,args)
    result=subprocess.run([sys.executable,"-c",GO08_BOOTSTRAP,str(project),str(launch)],cwd=project,capture_output=True,text=True,
                          env={**os.environ,"CUBLAS_WORKSPACE_CONFIG":":4096:8"},timeout=180)
    atomic_write_bytes(launch.with_suffix(".log"),(result.stdout+result.stderr).encode())
    if result.returncode:raise RuntimeError(f"Warmup worker failed; inspect {launch.with_suffix('.log')}")
    return json.loads((project/"reports"/result_file).read_text())

def validate_go_warmup_resume(project,data_file,device="cpu",model_config=None):
    project=Path(project);model_config=model_config or GoAHRConfig();data=json.loads(Path(data_file).read_text())
    config=GoWarmupConfig(epochs=2,batch_size=8,device=device,purpose="synthetic_smoke",checkpoint_steps=1)
    if 2*math.ceil(len(data["train"])/8)>16:raise ValueError("Use a small recovery fixture")
    suffix=uuid.uuid4().hex[:8];full_id=f"go08-smoke-full-{suffix}";resumed_id=f"go08-smoke-resumed-{suffix}"
    _go08_fresh_launch(project,full_id,data_file,config,model_config)
    part=_go08_fresh_launch(project,resumed_id,data_file,config,model_config,stop=1)
    assert part["optimizer_step"]==1
    end=_go08_fresh_launch(project,resumed_id,data_file,config,model_config)
    assert end["resumed_from"]==1
    a=strict_go_checkpoint(project,full_id);b=strict_go_checkpoint(project,resumed_id)
    compared=["model","optimizer","scheduler","scaler","rng","sampler","optimizer_step","config","metrics","best_dev_loss"]
    for key in compared:
        if not architecture_equal(a[key],b[key]):raise AssertionError(f"Exact warmup resume mismatch: {key}")
    try:load_go_ahr(project,resumed_id)
    except ValueError:pass
    else:raise AssertionError("Validation checkpoint accepted as strength candidate")
    return {"status":"PASS","device":device,"model_config":asdict(model_config),"source_hashes":go08_source_hashes(project),
            "full_run":full_id,"resumed_run":resumed_id,"interrupted_step":1,"final_optimizer_step":a["optimizer_step"],"compared":compared,
            "comparison":"bitwise equality","fresh_process_launches":3}


## Short model, search and target checks

These checks cover every head, legal policy masking, terminal perspective, independent final move validation, finite gradients, curriculum replay audit, invalid outcome rejection and bounded search. CUDA checks exercise both available GPUs; no long training is started. The recovery section below performs a few real optimizer steps on this small verified curriculum, labels those checkpoints `synthetic_smoke` and prevents them from being loaded as strength candidates.

In [ ]:
torch.set_num_threads(1)
GO_CONFIG08=GoAHRConfig()
WARMUP_PATH08,WARMUP_DATA08=make_go_warmup_curriculum(PROJECT)
assert verify_go_warmup_data(WARMUP_DATA08)
assert all({GoState.from_dict(r["state"]).to_play for r in WARMUP_DATA08[split]}=={BLACK,WHITE} for split in ("train","dev"))
assert {r["group_id"] for r in WARMUP_DATA08["train"]}.isdisjoint({r["group_id"] for r in WARMUP_DATA08["dev"]})
MODEL08=GoAHR(GO_CONFIG08)
EXAMPLES08=WARMUP_DATA08["train"][:4]
loss08,metrics08=go_learning_loss(MODEL08,EXAMPLES08);loss08.backward()
assert all(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in MODEL08.parameters())
state08=GoState.from_dict(EXAMPLES08[0]["state"])
MODEL08.eval();batch08=encode_go_batch([state08]);out08=MODEL08(batch08)
assert out08["policy_logits"].shape==(1,82) and out08["ownership"].shape==(1,81)
assert torch.all(out08["policy_logits"].softmax(-1)[~batch08["legal_mask"]]==0)
ending08=play(play(initial_state(9),81),81)
black08=terminal_go_targets(ending08,BLACK);white08=terminal_go_targets(ending08,WHITE)
assert black08["outcome_target"]==-white08["outcome_target"] and black08["score_target"]==-white08["score_target"]
try:terminal_go_targets(initial_state(9),BLACK)
except ValueError:pass
else:raise AssertionError("Nonterminal outcome accepted")
bad08=copy.deepcopy(EXAMPLES08[0]);bad08["terminal_targets_verified"]=False
try:go_learning_loss(MODEL08,[bad08])
except ValueError:pass
else:raise AssertionError("Unverified terminal target accepted")
search08=choose_go_ahr_move(state08,MODEL08,1.,GoAHRSearchConfig(max_depth=2,branch_width=2,max_nodes=24))
assert search08["verified_legal"] and search08["strategic_status"]=="UNKNOWN"
assert math.isclose(sum(search08["policy_target"]),1.,abs_tol=1e-6)
fallback08=choose_go_ahr_move(state08,MODEL08,0.)
assert fallback08["action"]==state08.pass_action and fallback08["fallback_pass"] and fallback08["status"]=="UNKNOWN"
terminal08=choose_go_ahr_move(ending08,MODEL08,1.)
assert terminal08["status"]=="TERMINAL" and terminal08["action"] is None
GPU_CHECKS08=[]
for index in range(min(2,torch.cuda.device_count())):
    gpu=GoAHR(GO_CONFIG08).to(f"cuda:{index}")
    loss,stats=go_learning_loss(gpu,EXAMPLES08);loss.backward()
    assert all(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in gpu.parameters())
    GPU_CHECKS08.append({"device":f"cuda:{index}","finite_gradients":True,"loss":float(loss)})
    del gpu
print({"model_parameters":sum(p.numel() for p in MODEL08.parameters()),"curriculum":WARMUP_DATA08["counts"],
       "loss_heads":metrics08,"search":{"action":search08["action"],"nodes":search08["nodes"],"elapsed_s":search08["elapsed_s"]},"gpu_checks":GPU_CHECKS08})

## Exact fresh-process optimizer recovery

Each device configuration runs an uninterrupted control, a process that exits after optimizer step 1, and a new process that restores and completes the run. Model, optimizer, scheduler, disabled AMP scaler, Python/NumPy/CPU/all-CUDA RNGs, permutation/cursor/epoch and metrics must match bit for bit. These are disposable validation checkpoints. Full warmup requires a matching source/model/device receipt.

One writer owns each run directory. Checkpoints are atomic, save every stage/epoch and at most 50 steps or 600 seconds, and retain latest two plus best. `global_step` is the monotonic checkpoint sequence; `optimizer_step` records updates. Best is selected by development loss and is still an unpromoted candidate. There is no replay buffer or persistent per-example recurrent state in this warmup trainer; notebook 09 saves replay and in-progress games separately.

In [ ]:
RESUME_CHECKS08=[]
for device in (["cpu","cuda:0"] if torch.cuda.is_available() else ["cpu"]):
    RESUME_CHECKS08.append(validate_go_warmup_resume(PROJECT,WARMUP_PATH08,device,GO_CONFIG08))
atomic_json(PROJECT/"reports/08_exact_resume_validation.json",{"status":"PASS","configurations":RESUME_CHECKS08})
REPORT08={"status":"PASS","source_hashes":go08_source_hashes(PROJECT),"model_config":asdict(GO_CONFIG08),
          "parameter_count":sum(p.numel() for p in MODEL08.parameters()),"curriculum":str(WARMUP_PATH08.relative_to(PROJECT)),
          "curriculum_sha256":file_sha256(WARMUP_PATH08),"counts":WARMUP_DATA08["counts"],"gpu_checks":GPU_CHECKS08,
          "exact_resume":RESUME_CHECKS08,"training_route":"go_supervised","strength":"NOT_EVALUATED",
          "search_check":{"verified_legal":True,"strategic_status":"UNKNOWN","nodes":search08["nodes"]},
          "checks":["legal_replay_targets","group_isolation","all_heads_and_gradients","exact_hard_mask","signed_perspectives","cutoff_no_outcome","search_legality","zero_budget_unknown"],
          "visual_inspection":"Saved output review and HTML structural inspection; graphical review unavailable"}
atomic_json(PROJECT/"reports/08_go_learning_validation.json",REPORT08)
print({"validation":"PASS","recovery_devices":[r["device"] for r in RESUME_CHECKS08],"fresh_processes":3*len(RESUME_CHECKS08),"strength":"NOT_EVALUATED"})

## Launch and resume initial Go learning

Default `AHR08_MODE=validate` runs the bounded validation above. Set `AHR08_MODE=train` to run the configured 12-epoch warmup on the verified generated curriculum; this is an initial representation-learning experiment, not a strong-player training claim. On this small dataset it is expected to take a few minutes. Long self-play is a separate launch in 09.

```text
AHR08_MODE=train AHR08_RUN_ID=go08-initial-warmup-v2 .venv/bin/python -m jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.kernel_name=ahr-go --ExecutePreprocessor.timeout=900 notebooks/08_go_specific_adaptation.ipynb
```

**Exact resume:** run this same command with the same run ID, curriculum, device and exported code. It detects the committed manifest, verifies hashes, restores the recorded optimizer step/epoch/cursor/RNG and reports `resumed_from`; a completed run does not repeat training. Invalid manifests/configuration are errors, never an implicit restart. Checkpoint files are in `checkpoints/go08-initial-warmup-v2/`; its manifest points to this section. The callable `run_go_warmup` permits explicit alternative configurations, which require matching recovery validation first.

Model loading for 09–11 uses `load_go_ahr(PROJECT, run_id, selection="best")`. Old CSP checkpoints are deliberately a different model schema. New Go checkpoints retain their training provenance and remain candidates until arena evaluation. The persistent boot disk's instance-deletion policy is unverified; before recycling/deleting the instance, copy and hash-verify model/data/notebook artifacts off-instance.

In [ ]:
MODE08=os.environ.get("AHR08_MODE","validate")
if MODE08=="train":
    RUN08=os.environ.get("AHR08_RUN_ID","go08-initial-warmup-v2")
    CONFIG08=GoWarmupConfig(device="cuda:0" if torch.cuda.is_available() else "cpu")
    RESULT08=run_go_warmup(PROJECT,RUN08,WARMUP_PATH08,CONFIG08,GO_CONFIG08)
    TRAINED08,META08=load_go_ahr(PROJECT,RUN08,device="cpu")
    atomic_json(PROJECT/"reports/08_warmup_results.json",{**RESULT08,"selected_checkpoint":META08,"strength":"NOT_EVALUATED"})
    print({"run_id":RUN08,"complete":RESULT08["complete"],"optimizer_steps":RESULT08["optimizer_step"],"selected_step":META08["optimizer_step"],"strength":"NOT_EVALUATED"})
elif MODE08=="validate":
    print("Short model/recovery validation complete. No full warmup or self-play launch in validate mode.")
else:raise ValueError("AHR08_MODE must be validate or train")